# Training and Evaluation for Paper dataset with varying number of LTN training examples

This notebook is to trace the behaviour of anomaly detection against the number of available LTN Anomalous examples

In [1]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices)
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print("GPU found")
    print("Memory growth set")
else:
    print("No GPU found")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set


In [2]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

import itertools

from sklearn import metrics


from april.anomalydetection import BINet
from april.anomalydetection.utils import label_collapse
from april.database import Evaluation
from april.paperevaluator import Evaluator
from april.fs import get_model_files
from april.fs import PLOT_DIR

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sqlalchemy.orm import Session
import scikit_posthocs as sp

from april.database import get_engine
from april.fs import PLOT_DIR
from april.utils import microsoft_colors, prettify_dataframe, cd_plot, get_cd
from april.enums import Base, Strategy, Heuristic

sns.set_style('white')
pd.set_option('display.max_rows', 50)
%config InlineBackend.figure_format = 'retina'


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set
Creating Evaluation table


In [3]:
dataset = "paper-0.3-1"
out_dir = PLOT_DIR / 'paper_evaluations'
eval_file = out_dir / 'paper_fraction_evaluations.pkl'
csv_file = out_dir / 'paper_fraction_evaluations.csv'
excel_file = out_dir / 'paper_fraction_evaluations.xlsx'
model_folder = r"D:\LTNcoder\.out\models"
db = r"D:\LTNcoder\.out\april.db"

# code to create out_dir if it does not exist
if not out_dir.exists():
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {out_dir}")
from april.utils import delete_all_files_in_folder, delete_evaluation_and_model_tables
delete_all_files_in_folder(model_folder)
delete_evaluation_and_model_tables(db)


Deleted all rows from Evaluation and Model tables.


# Training

In [4]:
def fit_and_save(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
    if ad_kwargs is None:
        ad_kwargs = {}
    if fit_kwargs is None:
        fit_kwargs = {}

    # Save start time
    start_time = arrow.now()

    # Dataset
    dataset = Dataset(dataset_name)

    # AD
    ad = ad(**ad_kwargs)

    # Train and save
    ad.fit(dataset, **fit_kwargs)
    file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
    model_file = ad.save(file_name)

    # Save end time
    end_time = arrow.now()

    # Cache result
    Evaluator(model_file.str_path).cache_result()

    # Calculate training time in seconds
    training_time = (end_time - start_time).total_seconds()

    # Write to database
    engine = get_engine()
    session = Session(engine)

    session.add(Model(creation_date=end_time.datetime,
                      algorithm=ad.name,
                      training_duration=training_time,
                      file_name=model_file.file,
                      training_event_log_id=EventLog.get_id_by_name(dataset_name),
                      training_host=socket.gethostname(),
                      hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
    session.commit()
    session.close()

    if isinstance(ad, NNAnomalyDetector):
        from keras.backend import clear_session
        clear_session()
    pass

In [5]:
# ads = [
#     dict(ad=PaperDAE, fit_kwargs=dict(epochs=1, batch_size=100)),
#     dict(ad=PaperLTN, fit_kwargs=dict(epochs=1, batch_size=100, epochs_ltn=1)),
#     dict(ad=PaperLTNFROZEN, fit_kwargs=dict(epochs=1, batch_size=100, epochs_ltn=1)),    
# ]
ads = [
    dict(ad=LTN_ROW_CLASS, fit_kwargs=dict(epochs=8, batch_size=100, epochs_ltn=4))
    for LTN_ROW_CLASS in ltn_row_classes
]
for ad in tqdm(ads, desc="Fitting ADs"):
    fit_and_save(dataset, **ad)


Fitting ADs:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1/8
35/35 [==============================] - 1s 12ms/step - loss: 0.1960 - accuracy: 0.0026 - val_loss: 0.0592 - val_accuracy: 0.0000e+00
Epoch 2/8
35/35 [==============================] - 0s 8ms/step - loss: 0.0190 - accuracy: 0.0361 - val_loss: 0.0044 - val_accuracy: 0.0000e+00
Epoch 3/8
35/35 [==============================] - 0s 8ms/step - loss: 0.0049 - accuracy: 0.1365 - val_loss: 0.0043 - val_accuracy: 0.0000e+00
Epoch 4/8
35/35 [==============================] - 0s 8ms/step - loss: 0.0047 - accuracy: 0.2087 - val_loss: 0.0043 - val_accuracy: 0.0000e+00
Epoch 5/8
35/35 [==============================] - 0s 8ms/step - loss: 0.0046 - accuracy: 0.2613 - val_loss: 0.0043 - val_accuracy: 0.0000e+00
Epoch 6/8
35/35 [==============================] - 0s 8ms/step - loss: 0.0046 - accuracy: 0.2849 - val_loss: 0.0043 - val_accuracy: 0.0000e+00
Epoch 7/8
35/35 [==============================] - 0s 8ms/step - loss: 0.0045 - accuracy: 0.3135 - val_loss: 0.0042 - val_accuracy: 0.0000e+0

In [6]:
print(AD)

{'binetv0': <class 'april.anomalydetection.binet.binet.BINetv0'>, 'binetv1': <class 'april.anomalydetection.binet.binet.BINetv1'>, 'binetv2': <class 'april.anomalydetection.binet.binet.BINetv2'>, 'binetv3': <class 'april.anomalydetection.binet.binet.BINetv3'>, 'likelihood': <class 'april.anomalydetection.boehmer.BoehmerLikelihoodAnomalyDetector'>, 'dae': <class 'april.anomalydetection.autoencoder.DAE'>, 'daeltn': <class 'april.anomalydetection.autoencoder.DAELTN'>, 'daeltnfrozen': <class 'april.anomalydetection.autoencoder.DAELTNFROZEN'>, 'likelihood+': <class 'april.anomalydetection.boehmer.LikelihoodPlusAnomalyDetector'>, 'naive': <class 'april.anomalydetection.bezerra.NaiveAnomalyDetector'>, 'naive+': <class 'april.anomalydetection.bezerra.NaivePlusAnomalyDetector'>, 'one-class-svm': <class 'april.anomalydetection.basic.OneClassSVM'>, 'p2pdae': <class 'april.anomalydetection.p2pdaeltn.P2PDAE'>, 'p2pdaeltn': <class 'april.anomalydetection.p2pdaeltn.P2PDAELTN'>, 'paperdae': <class 'ap

# Evaluation

In [7]:
heuristics = [h for h in Heuristic.keys() if h not in [Heuristic.DEFAULT, Heuristic.MANUAL, Heuristic.RATIO,
                                                       Heuristic.MEDIAN, Heuristic.MEAN]]
params = [(Base.SCORES, Heuristic.DEFAULT, Strategy.SINGLE), *itertools.product([Base.SCORES], heuristics, Strategy.keys())]

In [8]:
def _evaluate(params):
    # print(f"Evaluating {params}...")
    e, base, heuristic, strategy = params

    session = Session(get_engine())
    model = session.query(Model).filter_by(file_name=e.model_file.name).first()
    session.close()
    # print(f"Model {model} loaded from Session.")

    if Model is None:
        print(f"Models from session: {model}")
    # Generate evaluation frames
    y_pred = e.binarizer.binarize(base=base, heuristic=heuristic, strategy=strategy, go_backwards=False)
    y_true = e.binarizer.get_targets()

    evaluations = []
    for axis in [0, 1, 2]:
        # print(f"Evaluating axis {axis}...")
        for i, attribute_name in enumerate(e.dataset.attribute_keys):
            # print(f"Evaluating attribute {attribute_name}...")
            def get_evaluation(label, precision, recall, f1):
                return Evaluation(model_id=model.id, file_name=model.file_name,
                                  label=label, perspective=perspective, attribute_name=attribute_name,
                                  axis=axis, base=base, heuristic=heuristic, strategy=strategy,
                                  precision=precision, recall=recall, f1=f1)

            perspective = 'Control Flow' if i == 0 else 'Data'
            if i > 0 and not e.ad_.supports_attributes:
                # print(f"Skipping attribute {attribute_name} for model {model} as it does not support attributes.")
                evaluations.append(get_evaluation('Normal', 0.0, 0.0, 0.0))
                evaluations.append(get_evaluation('Anomaly', 0.0, 0.0, 0.0))
            else:
                # print(f"Evaluating attribute {attribute_name} for model {model}...")
                yp = label_collapse(y_pred[:, :, i:i + 1], axis=axis).compressed()
                yt = label_collapse(y_true[:, :, i:i + 1], axis=axis).compressed()
                p, r, f, _ = metrics.precision_recall_fscore_support(yt, yp, labels=[0, 1])
                evaluations.append(get_evaluation('Normal', p[0], r[0], f[0]))
                evaluations.append(get_evaluation('Anomaly', p[1], r[1], f[1]))

    return evaluations

def evaluate(model_name):
    # print(f"Evaluating {model_name}...")
    e = Evaluator(model_name)
    # print(f"{e} loaded.")

    _params = []
    for base, heuristic, strategy in params:
        if e.dataset.num_attributes == 1 and strategy in [Strategy.ATTRIBUTE, Strategy.POSITION_ATTRIBUTE]:
            continue
        if isinstance(e.ad_, BINet) and e.ad_.version == 0:
            continue
        if heuristic is not None and heuristic not in e.ad_.supported_heuristics:
            continue
        if strategy is not None and strategy not in e.ad_.supported_strategies:
            continue
        if base is not None and base not in e.ad_.supported_bases:
            continue
        _params.append([e, base, heuristic, strategy])

    return [_e for p in _params for _e in _evaluate(p)]

In [9]:
models = sorted([m.name for m in get_model_files() if m.p == 0.3])# and 'real' in m.name])
print(f"Available Models: {models}")
evaluations = []
for model in tqdm(models, desc='Evaluate'):
    e = evaluate(model)
    evaluations.append(e)
# Write to database
session = Session(get_engine())
for e in evaluations:
    session.bulk_save_objects(e)
    session.commit()
session.close()

Available Models: ['paper-0.3-1_paperltnfrozen-100_20250504-235612.434567', 'paper-0.3-1_paperltnfrozen-10_20250504-235101.457869', 'paper-0.3-1_paperltnfrozen-150_20250504-235756.629848', 'paper-0.3-1_paperltnfrozen-200_20250505-000052.669061', 'paper-0.3-1_paperltnfrozen-250_20250505-000354.129061', 'paper-0.3-1_paperltnfrozen-25_20250504-235243.009983', 'paper-0.3-1_paperltnfrozen-300_20250505-000645.097798', 'paper-0.3-1_paperltnfrozen-350_20250505-000951.047101', 'paper-0.3-1_paperltnfrozen-400_20250505-001258.391829', 'paper-0.3-1_paperltnfrozen-50_20250504-235426.368154']


Evaluate:   0%|          | 0/10 [00:00<?, ?it/s]

Loading model paper-0.3-1_paperltnfrozen-100_20250504-235612.434567 for event log paper-0.3-1
Evaluuation dataset shape: (669, 17, 2)
Loading model paper-0.3-1_paperltnfrozen-10_20250504-235101.457869 for event log paper-0.3-1
Evaluuation dataset shape: (669, 17, 2)
Loading model paper-0.3-1_paperltnfrozen-150_20250504-235756.629848 for event log paper-0.3-1
Evaluuation dataset shape: (669, 17, 2)
Loading model paper-0.3-1_paperltnfrozen-200_20250505-000052.669061 for event log paper-0.3-1
Evaluuation dataset shape: (669, 17, 2)
Loading model paper-0.3-1_paperltnfrozen-250_20250505-000354.129061 for event log paper-0.3-1
Evaluuation dataset shape: (669, 17, 2)
Loading model paper-0.3-1_paperltnfrozen-25_20250504-235243.009983 for event log paper-0.3-1
Evaluuation dataset shape: (669, 17, 2)
Loading model paper-0.3-1_paperltnfrozen-300_20250505-000645.097798 for event log paper-0.3-1
Evaluuation dataset shape: (669, 17, 2)
Loading model paper-0.3-1_paperltnfrozen-350_20250505-000951.047

In [10]:

session = Session(get_engine())
evaluations = session.query(Evaluation).all()
rows = []

for ev in tqdm(evaluations):
    # print(f"Evaluation: {ev}")
    m = ev.model
    # print(f"Model: {m}")
    el = ev.model.training_event_log
    # print(f"Event log: {el}")
    rows.append([m.file_name, m.creation_date, m.hyperparameters, m.training_duration, m.training_host, m.algorithm, 
                 el.name, el.base_name, el.percent_anomalies, el.number,
                 ev.axis, ev.base, ev.heuristic, ev.strategy, ev.label, ev.attribute_name, ev.perspective, ev.precision, ev.recall, ev.f1])
session.close()
columns = ['file_name', 'date', 'hyperparameters', 'training_duration', 'training_host', 'ad',
           'dataset_name', 'process_model', 'noise', 'dataset_id',
           'axis', 'base', 'heuristic', 'strategy', 'label', 'attribute_name', 'perspective', 'precision', 'recall', 'f1']
evaluation = pd.DataFrame(rows, columns=columns)

evaluation.to_pickle(eval_file)

  0%|          | 0/2880 [00:00<?, ?it/s]

In [11]:
synth_datasets = ['paper', 'p2p', 'small', 'medium', 'large', 'huge', 'gigantic', 'wide']
bpic_datasets = ['bpic12', 'bpic13', 'bpic15', 'bpic17']
anonymous_datasets = ['real']
datasets = synth_datasets + bpic_datasets + anonymous_datasets
dataset_types = ['Synthetic', 'Real-life']

h_ads = ads = [ad['ad'].__name__ for ad in ads]

heuristics = [r'$best$', r'$default$', r'$elbow_\downarrow$', r'$elbow_\uparrow$', 
              r'$lp_\leftarrow$', r'$lp_\leftrightarrow$', r'$lp_\rightarrow$']
print(ads)

['PaperLTNFROZEN-10', 'PaperLTNFROZEN-25', 'PaperLTNFROZEN-50', 'PaperLTNFROZEN-100', 'PaperLTNFROZEN-150', 'PaperLTNFROZEN-200', 'PaperLTNFROZEN-250', 'PaperLTNFROZEN-300', 'PaperLTNFROZEN-350', 'PaperLTNFROZEN-400']


In [12]:
evaluation = evaluation.query(f'ad in {ads} and label == "Anomaly"')

In [13]:
evaluation['perspective-label'] = evaluation['perspective'] + '-' + evaluation['label']
evaluation['attribute_name-label'] = evaluation['attribute_name'] + '-' + evaluation['label']
evaluation['dataset_type'] = 'Synthetic'
evaluation.loc[evaluation['process_model'].str.contains('bpic'), 'dataset_type'] = 'Real-life'
evaluation.loc[evaluation['process_model'].str.contains('real'), 'dataset_type'] = 'Real-life'

In [14]:
_filtered_evaluation = evaluation.query(f'ad in {h_ads} and (strategy == "{Strategy.ATTRIBUTE}"'
                                       f' or (strategy == "{Strategy.SINGLE}" and process_model == "bpic12")'
                                       f' or (strategy == "{Strategy.SINGLE}" and ad == "Naive+"))')

In [15]:
filtered_evaluation = _filtered_evaluation.query(f'heuristic == "{Heuristic.DEFAULT}"'
                                                 f' or (heuristic == "{Heuristic.LP_MEAN}" and ad not in {h_ads})'
                                                 f' or (heuristic == "{Heuristic.LP_LEFT}" and ad in {h_ads})'
                                                )

In [16]:
df = filtered_evaluation.query('axis == 0')
df = prettify_dataframe(df)
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name', 'perspective'])[['precision', 'recall', 'f1']].mean().reset_index()
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name'])[['precision', 'recall', 'f1']].mean().reset_index()
df['f1'] = 2 * df['recall'] * df['precision'] / (df['recall'] + df['precision'])

df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model', 'dataset_name'], values=['precision', 'recall', 'f1'])
df = df.fillna(0)
df = df.stack(1).stack(1).reset_index()
df.to_excel(str(out_dir / 'table.xlsx'), index=False)

# drop rows in column "axis" which have value "Attribute"
df = df.query('axis != "Attribute"')

# df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model'], values=['precision', 'recall', 'f1'], aggfunc=np.mean)

df.to_excel(str(excel_file), index=False)
df.to_csv(str(csv_file), index=False)
print(df.head(10))

C:\Users\devas\AppData\Local\Temp\ipykernel_26252\265154155.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()
C:\Users\devas\AppData\Local\Temp\ipykernel_26252\265154155.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()


   axis                  ad process_model dataset_name        f1  precision  \
0  Case   PaperLTNFROZEN-10         Paper  paper-0.3-1  0.581919   0.630435   
1  Case  PaperLTNFROZEN-100         Paper  paper-0.3-1  0.604340   0.617188   
2  Case  PaperLTNFROZEN-150         Paper  paper-0.3-1  0.557841   0.632159   
3  Case  PaperLTNFROZEN-200         Paper  paper-0.3-1  0.545271   0.614068   
4  Case   PaperLTNFROZEN-25         Paper  paper-0.3-1  0.610262   0.631579   
5  Case  PaperLTNFROZEN-250         Paper  paper-0.3-1  0.573994   0.632159   
6  Case  PaperLTNFROZEN-300         Paper  paper-0.3-1  0.503730   0.617816   
7  Case  PaperLTNFROZEN-350         Paper  paper-0.3-1  0.636217   0.629310   
8  Case  PaperLTNFROZEN-400         Paper  paper-0.3-1  0.660347   0.607018   
9  Case   PaperLTNFROZEN-50         Paper  paper-0.3-1  0.673116   0.632159   

     recall  
0  0.540336  
1  0.592017  
2  0.499160  
3  0.490336  
4  0.590336  
5  0.525630  
6  0.425210  
7  0.643277  
8  0